## Setup

### Load Modules

In [ ]:
%load_ext autoreload
%autoreload 2

#General Import
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
from os.path import join
from sklearn.model_selection import KFold
from matplotlib import colormaps as cmaps
from mne.filter import filter_data, resample
import scipy.stats as stats
import pandas as pd
from itertools import product
import xarray as xr
from scipy.signal import coherence, welch
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import re
import colorcet as cc
from wordfreq import word_frequency
from scipy.special import comb
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap

#ML Import
from sklearn.decomposition import PCA, FastICA, SparsePCA, FactorAnalysis, NMF
from sklearn.preprocessing import StandardScaler, scale
from sklearn.metrics import silhouette_score
from dtaidistance.preprocessing import differencing
import dtaidistance.clustering.kmeans as dtwkmeans
import dtaidistance as dta
import jPCA
import scipy.signal as signal
from statsmodels.tsa.stattools import grangercausalitytests
import tslearn
from statsmodels.stats.multitest import multipletests

#Electrophysiology Import
from spyeeg.models.TRF import TRFEstimator
from spyeeg.models.ERP import ERP_class
from spyeeg.utils import lag_matrix
import mne
import frites
from frites.simulations import sim_multi_suj_ephy
from frites.dataset import DatasetEphy
from frites.workflow import WfConnComod
from frites import set_mpl_style
import frites.conn as conn
from scipy.signal import welch
import spectral_connectivity 

#Graph Import
import networkx as nx
from matplotlib_venn import venn3, venn3_circles, venn2
import matplotlib.gridspec as gridspec

#Performance Import
import time
import psutil

#Local Import
from stats_utils import cliffs_delta, cohen_d
from nice_utils import estimate_loop_time, decorator_loop
from preprocessing_utils import mono_to_bipolar, select_channels, available_regions, delete_channels, adj_scale
from signal_utils import sparse_resample, lag_finder
from viz_utils import create_matshow_gif, create_collection_gif, _arrow3D
from graph_utils import new_add_edge


In [2]:
from scipy.ndimage import find_objects
from scipy.ndimage import label as scipylabel

def filter_binary(binary_array, min_cluster_size=4):
    """
    Finds clusters of True values in a binary array and returns a mask array
    where only clusters of size >= min_cluster_size are True. Useful to 
    filter out isolated points.
    
    Parameters
    ----------
    binary_array : ndarray
        1D array of booleans to filter
    min_cluster_size : int
        How many successive True values are necessary to define a cluster

    Returns
    -------
    cluster_mask : ndarray
        masking array corresponding to the filtering.
    """
    

    # Label all connected components
    labeled_array, num_features = scipylabel(binary_array)
    slices = find_objects(labeled_array)

    # Filter clusters by size
    cluster_mask = np.zeros_like(binary_array, dtype=bool)
    for s in slices:
        cluster_size = s[0].stop - s[0].start
        if cluster_size >= min_cluster_size:
            cluster_mask[s] = True

    return cluster_mask

### Load Features

In [ ]:
# Acoustic Regressors
fs = 100
new_path = 'C:/Users/D-CAP/Documents/GitHub/witching-star/regressors/selected_regs.pkl'
new_data = pickle.load(open(new_path, 'rb'))
data_fs = new_data['fs']
new_regressors = new_data['regs']
new_names = new_data['regs_name']
ratio = fs/data_fs
new_duration = int(new_regressors.shape[0] * ratio) + 1

new_resamp = []
for i in range(new_regressors.shape[1]):
    name = new_names[i]
    if name in ['Intensity', 'Envelope Oganian', 'Envelope Derivative TF', 'F0 Loudness', 'SpectralFlux Filtered', 'SpectralFlux not_filtered']:
        new_reg = mne.filter.resample(new_regressors[:,i], up=100, down=data_fs)[:new_duration]
    elif name in ['peakEnv_tf', 'Syllabe Onset', 'p-syl', 'Phono']:
        new_reg = new_regressors[:,i] - np.min(new_regressors[:,i])
        new_reg = sparse_resample(new_reg, new_fs = fs, current_fs = data_fs)[:new_duration]
    else:
        print('wut')
    new_resamp.append(new_reg)
new_resamp = np.asarray(new_resamp).T
regressors = new_resamp
regressors_name = new_data['regs_name']


In [ ]:
# Renyi2 Array

path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/renyi_array2.pickle"
renyi_data = pickle.load(open(path_renyi, 'rb'))
data_fs = renyi_data['fs']
X = np.roll(renyi_data['X'],4, axis=0)
onsets = np.where(X[:,5] >0)[0]
reg_renyi = np.zeros([regressors.shape[0], X.shape[1]])

for onset in onsets:
    new_onset = int(onset/data_fs*fs)
    for renyi_index in range(X.shape[1]):
        reg_renyi[new_onset, renyi_index] = X[onset,renyi_index]

regressors = np.hstack([regressors, reg_renyi])
regressors_name = regressors_name + renyi_data['names']
renyi_values_wrd = [float(renyi_name.split('renyi ')[1]) for renyi_name in renyi_data['names'][:20]]

## Load FIT Data

### Load FIT

In [ ]:
channel_selection = ['H','T']
#channel_selection = []
exclude = False
#channel_selection = ['H','T']
fit_group = dict()
pos_group = dict()
reg_color = []
reg_label_dict = {10: 'Temporal Phoneme Surprise', 12: 'Temporal Syllabic Surprise', 14:'Temporal Surprise',
                  16:'Semantic Surprise',17:'Semantic Entropy', 
                  18:'TxS Low',19:'TxS High', 20:'Scandal',21:'Entropy t-1',22:'Entropy t-2',
                  23:'TxE-1 Low',24:'TxE-1 high',25:'TxE-2 Low',26:'TxE-2 high',
                  28:'Phn Surprisal', 29:'Phn Entropy', 30: 'Phn Cross Entropy t-1,t', 32 : 'Phn Lexical Surprise',33 : 'Phn WR Surprise',34 : 'Phn WR Entropy', 35 : 'Phn Cam Surprise', 36 : 'Phn Cam Entropy',
                  37: 'Phn Sum S-E-1', 38: 'Phn Entropy t-1', 39: 'Phn Entropy t-2', 40: 'Phn Entropy t-3', 41: 'Phn Entropy t-4', 42: 'Phn Entropy t-5',43: 'Phn Entropy t-6',44: 'Phn Entropy t-7',
                  45: 'Phn Entropy t+1', 46:  'Phn Entropy t+2', 47:  'Phn Entropy t+3',
                  49: 'Phn Start Surprise', 50 : 'Phn Start Entropy', 53: 'Phn Start Lexical Surprise', 54: 'Phn Start WR Surprise', 55: 'Phn Start WR Entropy', 56: 'Phn Start Cam Surprise', 57: 'Phn Start Cam Entropy',
                  59: 'Phn End Surprise', 60 : 'Phn End Entropy', 63: 'Phn End Lexical Surprise', 64: 'Phn End WR Surprise', 65: 'Phn End WR Entropy', 66: 'Phn End Cam Surprise', 67: 'Phn End Cam Entropy',
                  69: 'Syllable Surprise', 70 : 'Syllable Entropy', 73: 'Syllable Lexical Surprise', 74: 'Syllable WR Surprise', 75: 'Syllable WR Entropy', 76: 'Syllable Cam Surprise', 77: 'Syllable Cam Entropy',
                  78: 'Syl Sum S-E-1', 79: 'Syllable Entropy t-1', 80: 'Syllable Entropy t-2', 81: 'Syllable Entropy t-3', 82: 'Syllable Entropy t-4', 83: 'Syllable Entropy t+1',
                  85: 'Syllable Surprise t-1', 86: 'Syllable Surprise t-2', 87: 'Syllable Surprise t-3', 88: 'Syllable Surprise t+1',
                  90: 'Surprise 2', 91: 'Renyi Entropy alpha = 0', 92: 'Renyi Entropy alpha = 0.5', 93: 'Renyi Entropy alpha = 1', 
                  94: 'Renyi Entropy alpha = 2', 95: 'Renyi Entropy alpha = 10', 96: 'Shannon Entropy t-1', 97: 'Shannon Entropy t-2', 98: 'Shannon Entropy t+1',
                  99: 'wrd_only surprise', 100: 'wrd_only_entropy', 101: 'wrd_only entropy t-1', 102: 'wrd_only entropy t-2' , 103: 'wrd_only entropy t-3',
                  105: 'wrd_only surprise, new', 106: 'wrd_only_entropy, new', 107: 'wrd_only renyi entropy',
                  108:  'wrd_only_entropy t-1, new', 109:  'wrd_only_entropy t-2, new', 110:  'wrd_only renyi entropy t-1', 111:  'wrd_only renyientropy t-2', 112:  'wrd_only renyientropy t+1', 
                  114: 'Surprisal',115: 'obvious entropy', 116:'surprising entropy', 117: 'Shannon Entropy', 118:'Lexical', 
                  119: 'Skewness', 120:'Kurtosis', 121: 'Fluctuation',
                  122: 'renyi shift1', 123: 'renyi shift1', 124: 'renyi shiftp1',
                  125: 'renyi 5e-6', 126:'renyi 0.16', 127: 'renyi 0.32', 128: 'renyi 0.47', 129:'renyi 0.62', 130: 'renyi 0.78', 131: 'renyi 0.95', 
                  132:'renyi 1.1', 133: 'renyi 1.26', 134: 'renyi 1.42', 135:'renyi 1.57', 136: 'renyi 1.74', 137: 'renyi 1.90', 138:'renyi 2.05: Collision Entropy', 139: 'renyi 2.21',
                  140:'renyi 2.36', 141:'renyi 2.52', 142:'renyi 2.68', 143:'renyi 2.84', 144:'renyi 3',
                  146: 'temporal phonemic surprise', 147: 'temporal phonemic entropy', 149: 'temporal syllabic surprise', 150: 'temporal syllabic entropy',152: 'Temporal Surprise', 153: 'temporal word entropy',
                  154: 'temporal phonemic empty entropy',155: 'temporal syllabic empty entropy',156: 'temporal word empty entropy',
                  158: 'Min Surprise', 159: 'Wasserstein', 160: 'Refresh Entropy', 161: 'KL Divergence',
                  162: 'phn renyi 0.05', 163:'phn renyi 0.18', 164: 'phn renyi 0.32', 165: 'phn renyi 0.46', 166:'phn renyi 0.60', 167: 'phn renyi 0.74', 168: 'phn renyi 0.88', 
                  169:'phn renyi 1.02', 170: 'phn renyi 1.18', 171: 'phn renyi 1.32', 172:'phn renyi 1.46', 173: 'phn renyi 1.6', 174: 'phn renyi 1.72', 175:'phn renyi 1.86', 176: 'phn renyi 2',
                  177:'phn renyi 2.14', 178:'phn renyi 2.35', 179:'phn renyi 2.57', 180:'phn renyi 2.78', 181:'phn renyi 3',
                  182: 'syl renyi 0.05', 183:'syl renyi 0.18', 184: 'syl renyi 0.32', 185: 'syl renyi 0.46', 186:'syl renyi 0.60', 187: 'syl renyi 0.74', 188: 'syl renyi 0.88', 
                  189:'syl renyi 1.02', 190: 'syl renyi 1.18', 191: 'syl renyi 1.32', 192:'syl renyi 1.46', 193: 'syl renyi 1.6', 194: 'syl renyi 1.72', 195:'syl renyi 1.86', 196: 'syl renyi 2',
                  197:'syl renyi 2.14', 198:'syl renyi 2.35', 199:'syl renyi 2.57', 200:'syl renyi 2.78', 201:'syl renyi 3',
                  203: 'phn entropy token', 204: 'phn fluctuation token', 205: 'phn kl divergence token', 206: 'phn cross entropy token',
                  208: 'syl entropy token', 209: 'syl fluctuation token', 210: 'syl kl divergence token', 211: 'syl cross entropy token',
                  212: 'renyi2 5e-6', 213:'renyi2 0.16', 214: 'renyi2 0.32', 215: 'renyi2 0.47', 216:'renyi2 0.62', 217: 'renyi2 0.78', 218: 'renyi2 0.95', 
                  219:'renyi2 1.1', 220: 'renyi2 1.26', 221: 'renyi2 1.42', 222:'renyi2 1.57', 223: 'renyi2 1.74', 224: 'renyi2 1.90', 225:'renyi2 2.05: Collision Entropy', 226: 'renyi2 2.21',
                  227:'renyi2 2.36', 228:'renyi2 2.52', 229:'renyi2 2.68', 230:'renyi2 2.84', 231:'renyi2 3', 
                  232: 'Silence',
                  233: 'Dispersion', 234: 'renyi WRD 0.016',235: 'renyi WRD 0.0263',236: 'renyi WRD 0.0428',237: 'renyi WRD 0.0695',238: 'renyi WRD 0.1128',239: 'renyi WRD 0.1832',
                  240: 'renyi WRD 0.2976',241: 'renyi WRD 0.4832',242: 'renyi WRD 0.7847',243: 'renyi WRD 1.2742',244: 'renyi WRD 2.0691',245: 'renyi WRD 3.3598',246: 'renyi WRD 5.4555',
                  247: 'renyi WRD 8.8586',248: 'renyi WRD 14.384',249: 'renyi WRD 23.357',250: 'renyi WRD 37.926',251: 'renyi WRD 61.584',252: 'renyi WRD 100.0',
                  253: 'Strength',254: 'Min2Entro',255: 'MaxEntro',256: 'Max2Entro',
                  257: 'Shannon Entropy',258: 'Surprisal',259: 'Fluctuation',
                  260: 'MaxRenyi WRD 0.01', 261: 'MaxRenyi WRD 0.016',262: 'MaxRenyi WRD 0.0263',263: 'MaxRenyi WRD 0.0428',264: 'MaxRenyi WRD 0.0695',265: 'MaxRenyi WRD 0.1128',266: 'MaxRenyi WRD 0.1832',
                  267: 'MaxRenyi WRD 0.2976',268: 'MaxRenyi WRD 0.4832',269: 'MaxRenyi WRD 0.7847',270: 'MaxRenyi WRD 1.2742',271: 'MaxRenyi WRD 2.0691',272: 'MaxRenyi WRD 3.3598',273: 'MaxRenyi WRD 5.4555',
                  274: 'MaxRenyi WRD 8.8586',275: 'MaxRenyi WRD 14.384',276: 'MaxRenyi WRD 23.357',277: 'MaxRenyi WRD 37.926',278: 'MaxRenyi WRD 61.584',279: 'MaxRenyi WRD 100.0',
                  280: 'Max2Renyi WRD 0.01', 281: 'Max2Renyi WRD 0.016',282: 'Max2Renyi WRD 0.0263',283: 'Max2Renyi WRD 0.0428',284: 'Max2Renyi WRD 0.0695',285: 'Max2Renyi WRD 0.1128',286: 'Max2Renyi WRD 0.1832',
                  287: 'Max2Renyi WRD 0.2976',288: 'Max2Renyi WRD 0.4832',289: 'Max2Renyi WRD 0.7847',290: 'Max2Renyi WRD 1.2742',291: 'Max2Renyi WRD 2.0691',292: 'Max2Renyi WRD 3.3598',293: 'Max2Renyi WRD 5.4555',
                  294: 'Max2Renyi WRD 8.8586',295: 'Max2Renyi WRD 14.384',296: 'Max2Renyi WRD 23.357',297: 'Max2Renyi WRD 37.926',298: 'Max2Renyi WRD 61.584',299: 'Max2Renyi WRD 100.0',
                  300: 'Max3Renyi WRD 0.01', 301: 'Max3Renyi WRD 0.016',302: 'Max3Renyi WRD 0.0263',303: 'Max3Renyi WRD 0.0428',304: 'Max3Renyi WRD 0.0695',305: 'Max3Renyi WRD 0.1128',306: 'Max3Renyi WRD 0.1832',
                  307: 'Max3Renyi WRD 0.2976',308: 'Max3Renyi WRD 0.4832',309: 'Max3Renyi WRD 0.7847',310: 'Max3Renyi WRD 1.2742',311: 'Max3Renyi WRD 2.0691',312: 'Max3Renyi WRD 3.3598',313: 'Max3Renyi WRD 5.4555',
                  314: 'Max3Renyi WRD 8.8586',315: 'Max3Renyi WRD 14.384',316: 'Max3Renyi WRD 23.357',317: 'Max3Renyi WRD 37.926',318: 'Max3Renyi WRD 61.584',319: 'Max3Renyi WRD 100.0',
                  320: 'Max4Renyi WRD 0.01', 321: 'Max4Renyi WRD 0.016',322: 'Max4Renyi WRD 0.0263',323: 'Max4Renyi WRD 0.0428',324: 'Max4Renyi WRD 0.0695',325: 'Max4Renyi WRD 0.1128',326: 'Max4Renyi WRD 0.1832',
                  327: 'Max4Renyi WRD 0.2976',328: 'Max4Renyi WRD 0.4832',329: 'Max4Renyi WRD 0.7847',330: 'Max4Renyi WRD 1.2742',331: 'Max4Renyi WRD 2.0691',332: 'Max4Renyi WRD 3.3598',333: 'Max4Renyi WRD 5.4555',
                  334: 'Max4Renyi WRD 8.8586',335: 'Max4Renyi WRD 14.384',336: 'MaxRenyi WRD 23.357',337: 'Max4Renyi WRD 37.926',338: 'MaxRenyi WRD 61.584',339: 'Max4Renyi WRD 100.0',
                  378: 'Surprisal offset',
                  620: 'Surprisal t-1',
                  1000: 'New SxE', 1001: 'New SxT', 1002: 'New ExT',
                  2114: 'Categorical Surprise',2116: 'Categorical Renyi Entropy (sum)', 2118: 'Categorical Renyi Entropy', 2125: 'Categorical E+S', 2126: 'Categorical E*S', 2127: 'Categorical max(E,S)'}
reg_color = []

subjects_indices = np.arange(len(data_bipolar))
regressors_list = [258,233,253]
reg_color = []
reg_color_dict = dict()
reg_label = []


for reg_index, reg in enumerate(regressors_list):
    reg_label.append(reg_label_dict[reg])
    color_value = int(reg_index/len(regressors_list)*240)
    #color_qual = int(reg_index/len(regressors_list)*(len(colormap) - 10) + 10)
    color_qual = int(reg_index)
    reg_color.append(cm.brg(color_value))
    #reg_color.append(cm.tab20b_r(color_qual))
    #reg_color.append(colormap[color_qual])
    reg_color_dict[reg] = reg_color[reg_index]

regressors_clusters = list(np.arange(233,253)) + [258]
montage_choice = 'bipo'
montage = 'bipo'
clustering = 'NMF6'
apply_baseline = False
baseline_str = (not apply_baseline) * 'no_baseline'
#clustering = 'Venn'
regressors_str = '_'.join(np.asarray(regressors_clusters).astype('str'))
avg = 'avg_' #'avg_' or '' or ''avg'
cat = '' #'cat_' or ''
channel_selection_join = ''.join(channel_selection)
for subject_index, subject_id in enumerate(data_bipolar):
    fit_group[subject_id] = dict()
    for regressor_index in regressors_list:
        eeg = data_subject[subject_id]
        channels = channels_subject[subject_id]
        locations = locations_subject[subject_id]
        target_name = clustering + '_' + montage_choice + '_' +  str(subject_id) + '_' + regressors_str
        filename = 'FIT/FIT4_' + baseline_str + montage + '_avg_' + target_name + '_reg' + str(regressor_index) + '_sub' + str(subject_id) + '.nc'
        with xr.open_dataset(filename) as ds:
            data = ds
        fit = data['FIT']
        fit_conn = frites.conn.conn_reshape_directed(fit, sep = '->')

        fit_group[subject_id][regressor_index] = {'timearray':fit, 'connectivity':fit_conn}


c:\users\d-cap\documents\github\frites\frites\conn\conn_utils.py:520: FutureWarning: the `pandas.MultiIndex` object(s) passed as 'roi' coordinate(s) or data variable(s) will no longer be implicitly promoted and wrapped into multiple indexed coordinates in the future (i.e., one coordinate for each multi-index level + one dimension coordinate). If you want to keep this behavior, you need to first wrap it explicitly using `mindex_coords = xarray.Coordinates.from_pandas_multiindex(mindex_obj, 'dim')` and pass it as coordinates, e.g., `xarray.Dataset(coords=mindex_coords)`, `dataset.assign_coords(mindex_coords)` or `dataarray.assign_coords(mindex_coords)`.
  da[axis] = pd.MultiIndex.from_arrays(
c:\users\d-cap\documents\github\frites\frites\conn\conn_utils.py:520: FutureWarning: the `pandas.MultiIndex` object(s) passed as 'roi' coordinate(s) or data variable(s) will no longer be implicitly promoted and wrapped into multiple indexed coordinates in the future (i.e., one coordinate for each mu

### Select Cluster

#### Choose Cluster

In [ ]:
montage_choice = 'bipo'
clustering = 'NMF6'

cluster_group = dict()
for subject_index, subject_id in enumerate(fit_group):
    regressors_str = '_'.join(np.asarray(regressors_clusters).astype('str'))
    filename = 'MI_cluster/' + clustering + '_' + str(subject_id) + '_' + regressors_str + '.pickle'
    filename = 'MI_cluster/' + clustering + '_' + montage_choice + '_' + str(subject_id) + '_' + regressors_str + '.pickle'
    with open(filename, 'rb') as file:
        cluster_group[subject_id] = pickle.load(file)

n_clusters = len(cluster_group[subject_id])
n_interactions = int(comb(n_clusters,2))
edges_list = []
for cluster_1 in range(n_clusters):
    for cluster_2 in range(n_clusters):
        edges_list.append(str(cluster_1) + '->' + str(cluster_2))    

In [8]:
cluster_mi = dict()
for subject_index, subject_id in enumerate(fit_group):
    regressors_str = '_'.join(np.asarray(regressors_clusters).astype('str'))
    filename = 'MI_cluster/' + clustering + '_fulldata_' + montage_choice + '_' + str(subject_id) + '_' + regressors_str + '.pickle'
    with open(filename, 'rb') as file:
        cluster_mi[subject_id] = pickle.load(file)

#### Cluster Fit data

In [9]:
def check_names(channel_name, name_a, name_b, order = False, separator = '-'):
    part1, part2 = channel_name.split(separator)
    if not order:
        return (part1 in name_a and part2 in name_b) or (part1 in name_b and part2 in name_a)
    else:
        return (part1 in name_a and part2 in name_b)

In [10]:
fit_cluster = dict()

for subject_index, subject_id in enumerate(fit_group):
    fit_cluster[subject_id] = dict()
    for regressor_index, regressor_id in enumerate(regressors_list):
        fit_cluster[subject_id][regressor_id] = dict()
        for cluster_1 in range(n_clusters):
            for cluster_2 in range(n_clusters):
                cluster_edge = str(cluster_1) + '->' + str(cluster_2)
                rois = fit_group[subject_id][regressor_id]['timearray'].roi.values
                roi1_find = cluster_group[subject_id][cluster_1]
                roi2_find = cluster_group[subject_id][cluster_2]
                if len(roi1_find) > 0 and len(roi2_find) > 0:
                    ds_fit_array = fit_group[subject_id][regressor_id]['timearray']
                    rois = ds_fit_array.roi.values
                    mask = np.array([check_names(name, roi1_find, roi2_find, order = True, separator = '->') for name in rois])
                    filtered_fit_array = ds_fit_array.sel(roi = ds_fit_array.roi[mask])

                    regex1 = '|'.join(roi1_find)
                    regex2 = '|'.join(roi2_find)
                    mask_source = fit_group[subject_id][regressor_id]['connectivity']['sources'].str.contains(regex1)
                    mask_target = fit_group[subject_id][regressor_id]['connectivity']['targets'].str.contains(regex2)
                    filtered_fit_conn = fit_group[subject_id][regressor_id]['connectivity'].sel(sources = fit_group[subject_id][regressor_id]['connectivity']['sources'][mask_source],
                                                                                              targets = fit_group[subject_id][regressor_id]['connectivity']['targets'][mask_target])

                    fit_cluster[subject_id][regressor_id][cluster_edge] = {'timearray':filtered_fit_array, 'connectivity': filtered_fit_conn}

### Helpers Dictionary

In [12]:
timecourse_array = ds.times 
delaycourse_array = ds.delays

## Timearray Analysis

### Make meta subject

In [18]:
meta_fit = dict()
edge_choices = ['0->1', '1->0']

for subject_index, subject_id in enumerate(fit_cluster.keys()):
    for reg in regressors_list:
        if not reg in meta_fit:
            print(reg)
            meta_fit[reg] = {'fit' : dict(), 'roi' : dict()}
        for edge in fit_cluster[subject_id][reg]:
            if edge in edge_choices:
                if not edge in meta_fit[reg]['fit']:
                    meta_fit[reg]['fit'][edge], meta_fit[reg]['roi'][edge] = [],[]
                fit_data = fit_cluster[subject_id][reg][edge]['timearray'].data 

                meta_fit[reg]['fit'][edge] += (list(fit_data))
                meta_fit[reg]['roi'][edge] += (list(fit_cluster[subject_id][reg][edge]['timearray']['roi'].data))

for reg in meta_fit:
    for info in meta_fit[reg]:
        for edge in meta_fit[reg][info]:
            meta_fit[reg][info][edge] = np.asarray(meta_fit[reg][info][edge])
            if info == 'fit':
                A = meta_fit[reg][info][edge]
                indices_triangle_x, indices_triangle_y = np.triu_indices(A.shape[1], k=A.shape[2] - A.shape[1] - 7, m=A.shape[2])
                A[:,indices_triangle_x,A.shape[2] - indices_triangle_y] = 0
                meta_fit[reg][info][edge] = A

233
253
258


In [19]:
for reg in meta_fit:
    for edge in meta_fit[reg]['fit']:
        plt.matshow(meta_fit[reg]['fit'][edge].mean(0))

In [22]:
%matplotlib qt

p_thres = 0.05
start_baseline = 0
end_baseline = 20
baseline_samples = list(np.arange(timecourse_array.shape[0]))[start_baseline:end_baseline]
reg_std = 1e-15
reg_mean = 1
past_control = np.arange(0,30)
min_cluster_size = 10
normalization = False

fig, ax = plt.subplots(figsize = (7,5), sharex = False, sharey = True)

colors = ['b','r', 'green']

edge = '0->1'

for reg_index in [0,1]:
    reg = regressors_list[reg_index]
    data = meta_fit[reg]['fit'][edge][:,past_control,:].sum(1)
    ax.plot(timecourse_array,np.mean(data,axis=0), color = colors[reg_index], label = reg_label[reg_index], linewidth = 2.5)
    ax.fill_between(timecourse_array,
                        np.nanmean(data,axis=0) + np.nanstd(data,axis=0)/np.sqrt(data.shape[0]), 
                        np.nanmean(data,axis=0)- np.nanstd(data,axis=0)/np.sqrt(data.shape[0]), 
                        color = colors[reg_index], alpha = 0.3)
    ax.spines[['right', 'top']].set_visible(0)
    ax.spines[['bottom', 'left']].set_linewidth(2)
    ax.set_ylabel('FIT (bits)', size = 20)
    ax.set_xlabel('Time (s)', size = 20)
    ax.tick_params(width=3, labelsize = 16)
    ax.yaxis.get_offset_text().set_fontsize(16)  
    ax.set_xlim(-0.2,1.25)
    ax.set_xticks([0,0.5,1])

### Preprocess Data

#### Build Dictionary

In [13]:
meta_fit = dict()
edge_choices = ['0->1','1->0', '0->0', '1->1']

for subject_index, subject_id in enumerate(fit_cluster.keys()):
    for reg in regressors_list:
        if not reg in meta_fit:
            print(reg)
            meta_fit[reg] = {'fit' : dict(), 'roi' : dict()}
        for edge in fit_cluster[subject_id][reg]:
            if edge in edge_choices:
                if not edge in meta_fit[reg]['fit']:
                    meta_fit[reg]['fit'][edge], meta_fit[reg]['roi'][edge] = [],[]
                fit_data = fit_cluster[subject_id][reg][edge]['timearray'].data 

                meta_fit[reg]['fit'][edge] += (list(fit_data))
                meta_fit[reg]['roi'][edge] += (list(fit_cluster[subject_id][reg][edge]['timearray']['roi'].data))

258
233
253


In [14]:

for reg in meta_fit:
    for info in meta_fit[reg]:
        for edge in meta_fit[reg][info]:
            meta_fit[reg][info][edge] = np.asarray(meta_fit[reg][info][edge])
            if info == 'fit':
                A = meta_fit[reg][info][edge]
                indices_triangle_x, indices_triangle_y = np.triu_indices(A.shape[1], k=A.shape[2] - A.shape[1] - 7, m=A.shape[2])
                if apply_baseline:
                    A[:,indices_triangle_x,A.shape[2] - indices_triangle_y] = 0
                meta_fit[reg][info][edge] = A

In [15]:
for reg in meta_fit:
    fit_net = dict()
    fit_netroi = dict()
    fit_values = meta_fit[reg]['fit']
    fit_roi = meta_fit[reg]['roi']
    for edge in fit_values:
            node1, node2 = edge.split('->')[0], edge.split('->')[1]
            edge1, edge2 = node1 + ('->') + node2,  node2 + ('->') + node1 
            edge_net1, edge_net2 = node1 + ('<->') + node2, node2 + ('<->') + node1
            fit1, fit2 = fit_values[edge1], fit_values[edge2]
            roi1, roi2 = fit_roi[edge1], fit_roi[edge2]
            if not (edge_net1 in fit_net or edge_net2 in fit_net):
                  fit_net[edge_net1] = np.zeros(fit1.shape)
                  fit_netroi[edge_net1] = []
                  for index1 in range(fit_roi[edge].shape[0]):
                        fit_chan1 = fit1[index1]
                        name_chan1 = roi1[index1]
                        chan1a, chan1b = name_chan1.split('->')[0], name_chan1.split('->')[1]
                        name_chan2 = chan1b + '->' + chan1a
                        index2 = np.where(roi2 == name_chan2)[0][0]
                        fit_chan2 = fit2[index1]
                        net = fit_chan1 - fit_chan2
                        name_chan_net = chan1a + '<->' + chan1b
                        fit_net[edge_net1][index1] = net
                        fit_netroi[edge_net1].append(name_chan_net)
    meta_fit[reg]['net_fit'] = fit_net
    meta_fit[reg]['net_roi'] = fit_roi               
                        
net_edges_list = list(fit_net.keys())

In [57]:
#for reg in meta_fit:
for reg in [258]:
    for edge in meta_fit[reg]['fit']:
        plt.matshow(meta_fit[reg]['fit'][edge].mean(0), origin = 'upper', cmap = 'jet')

In [59]:
#for reg in meta_fit:
for reg in [258]:
    for edge in meta_fit[reg]['net_fit']:
        plt.matshow(meta_fit[reg]['net_fit'][edge].mean(0), origin = 'upper', cmap = 'jet')

### Net Receiving Cluster

#### Statistics

In [39]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from matplotlib.collections import LineCollection


def colored_line(x, y, c, ax, **lc_kwargs):
    """
    Plot a line with a color specified along the line by a third value.

    It does this by creating a collection of line segments. Each line segment is
    made up of two straight lines each connecting the current (x, y) point to the
    midpoints of the lines connecting the current point with its two neighbors.
    This creates a smooth line with no gaps between the line segments.

    Parameters
    ----------
    x, y : array-like
        The horizontal and vertical coordinates of the data points.
    c : array-like
        The color values, which should be the same size as x and y.
    ax : Axes
        Axis object on which to plot the colored line.
    **lc_kwargs
        Any additional arguments to pass to matplotlib.collections.LineCollection
        constructor. This should not include the array keyword argument because
        that is set to the color argument. If provided, it will be overridden.

    Returns
    -------
    matplotlib.collections.LineCollection
        The generated line collection representing the colored line.
    """
    if "array" in lc_kwargs:
        warnings.warn('The provided "array" keyword argument will be overridden')

    # Default the capstyle to butt so that the line segments smoothly line up
    default_kwargs = {"capstyle": "butt"}
    default_kwargs.update(lc_kwargs)

    # Compute the midpoints of the line segments. Include the first and last points
    # twice so we don't need any special syntax later to handle them.
    x = np.asarray(x)
    y = np.asarray(y)
    x_midpts = np.hstack((x[0], 0.5 * (x[1:] + x[:-1]), x[-1]))
    y_midpts = np.hstack((y[0], 0.5 * (y[1:] + y[:-1]), y[-1]))

    # Determine the start, middle, and end coordinate pair of each line segment.
    # Use the reshape to add an extra dimension so each pair of points is in its
    # own list. Then concatenate them to create:
    # [
    #   [(x1_start, y1_start), (x1_mid, y1_mid), (x1_end, y1_end)],
    #   [(x2_start, y2_start), (x2_mid, y2_mid), (x2_end, y2_end)],
    #   ...
    # ]
    coord_start = np.column_stack((x_midpts[:-1], y_midpts[:-1]))[:, np.newaxis, :]
    coord_mid = np.column_stack((x, y))[:, np.newaxis, :]
    coord_end = np.column_stack((x_midpts[1:], y_midpts[1:]))[:, np.newaxis, :]
    segments = np.concatenate((coord_start, coord_mid, coord_end), axis=1)

    lc = LineCollection(segments, **default_kwargs)
    lc.set_array(c)  # set the colors of each segment

    return ax.add_collection(lc)

In [ ]:
from scipy.sparse import lil_matrix

data_net = meta_fit[258]['net_fit']['0<->1'][:,25:,:] 
data_mean = data_net.mean(0)
n_points = data_mean.shape[0] * data_mean.shape[1]
adjacency = lil_matrix((n_points, n_points))

def idx(i, j):
    return i * data_mean.shape[1] + j

# Build adjacency
for i in range(data_mean.shape[0]):
    for j in range(data_mean.shape[1]):
        current = idx(i, j)
        for di in [-1, 0, 1]:
            for dj in [-1, 0, 1]:
                if di == 0 and dj == 0:
                    continue  # skip self
                ni, nj = i + di, j + dj
                if 0 <= ni < data_mean.shape[0] and 0 <= nj < data_mean.shape[1]:
                    neighbor = idx(ni, nj)
                    adjacency[current, neighbor] = 1

# Convert to CSR for efficient computations
adjacency = adjacency.tocsr()

In [49]:
from mne.stats import permutation_cluster_1samp_test

n_permumations = 1000

T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(
    data_net, n_permutations=n_permumations, tail=0, n_jobs=-1, adjacency=None, threshold = None#, adjacency=adjacency
)

Using a threshold of 1.962451
stat_fun(H1): min=nan max=nan
Running initial clustering …
Found 1240 clusters


C:\Users\D-CAP\AppData\Local\Temp\ipykernel_12200\2702762618.py:5: RuntimeWarning: Provided stat_fun does not treat variables independently. Setting buffer_size to None.
  T_obs, clusters, cluster_p_values, H0 = permutation_cluster_1samp_test(


  0%|          | Permuting : 0/999 [00:00<?,       ?it/s]

In [50]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# Define custom color dictionary, less grey influence at center
cdict = {
    'red':   [(0.0, 0.0, 0.0),
              (0.5, 0.75, 0.75),  # light grey component
              (1.0, 1.0, 1.0)],

    'green': [(0.0, 0.0, 0.0),
              (0.5, 0.75, 0.75),
              (1.0, 0.0, 0.0)],

    'blue':  [(0.0, 1.0, 1.0),
              (0.5, 0.75, 0.75),
              (1.0, 0.0, 0.0)]
}

# Create the colormap
custom_seismic_softgrey = mcolors.LinearSegmentedColormap('CustomSeismicSoftGrey', cdict)

# Example visualization
plt.imshow(np.linspace(-1, 1, 256).reshape(1, -1), aspect='auto', cmap=custom_seismic_softgrey)
plt.colorbar()
plt.title('Seismic-like with Soft Grey Center')
plt.show()


#### Main Figure

In [51]:
import matplotlib.ticker as ticker
%matplotlib qt

fig = plt.figure(figsize=(6.5,5.9))
gs = gridspec.GridSpec(5, 3, width_ratios=[4, 1,0.2], height_ratios=[2,0.5, 4, 2, 2],
                       wspace=0.02, hspace=0.02)

data_mean = data_net.mean(0)
data_pval = np.ones(data_mean.shape) * 0
alphas = np.zeros(data_mean.shape)
wrd_distrib = np.load('wrd_distrib.npy')

# Main imshow
ax = fig.add_subplot(gs[2,0])

for cluster_index, pval in enumerate(cluster_p_values):
    if pval <= 0.05:
        data_pval[clusters[cluster_index][0],clusters[cluster_index][1]] = 1
        alphas[clusters[cluster_index][0],clusters[cluster_index][1]] = 1
data_signif = data_mean*data_pval
data_net_filt = data_net * data_pval[np.newaxis,:,:]
vmin, vmax = -np.abs(data_mean).max(), np.abs(data_mean).max()
#vmin, vmax = -np.nanmax(np.abs(T_obs)), np.nanmax(np.abs(T_obs))
#ax.imshow(data_mean, cmap = "Greys", alpha = 0.5)
#ax.imshow(data_signif, cmap = "seismic", origin = 'upper', vmin = vmin, vmax = vmax, alpha = alphas, aspect = 'equal')

#ax.imshow(T_obs, cmap = "seismic", alpha = 1, vmin = vmin, vmax = vmax)
im = ax.imshow(data_mean, cmap = "seismic", alpha = 1, vmin = vmin, vmax = vmax)
contour = ax.contour(data_pval, levels=[0], colors='k', linewidths=2, origin='lower')
ax.plot([193,193+75],[75,0],c = 'k', ls = '--', lw = 2)
#ax.plot([142,217],[75,0],c = 'k', ls = '--', lw = 2)
ax.plot([150,150+75],[75,0],c = 'k', ls = '--', lw = 2)
ax.plot([115,115+75],[75,0],c = 'k', ls = '--', lw = 2)
ax.axvline(188,c = 'k', ls = '--', lw = 2)
ax.axvline(258,c = 'k', ls = '--', lw = 2)
ax.axvline(150,c = 'k', ls = '--', lw = 2)
#ax.axvline(np.argmin(np.abs(timecourse_array.data))+1, color = 'k', ls = '--', lw = 2)
#ax.axvline(np.argmin(np.abs(timecourse_array.data - np.percentile(wrd_distrib,95)))+1,color = 'k', lw = 2, ls = '--')
ax.spines[['bottom', 'left','top','right']].set_linewidth(2)
ax.set_ylabel('Delays (s)', size = 20)
ax.set_xlabel('          Time (s)', size = 20)
ax.tick_params(width=3, labelsize = 16)
ax.yaxis.get_offset_text().set_fontsize(16)  

ax.set_xticks([100,150,200,250])
ax.set_xticklabels([-0.5,0,0.5,1])
ax.set_xlim(100,270)

ax.set_yticks([25,75])
ax.set_ylim(75,0)
ax.set_yticklabels([0.5,0])


# Top plot: mean along x
ax_top = fig.add_subplot(gs[0,0])
data_top = data_net_filt
data_top = data_net[:,:,:]

data_receiver = data_top.mean(0).sum(0)
data_receiver_color = (data_receiver - np.min(data_receiver)) / (np.max(data_receiver) - np.min(data_receiver))
#ax_top.scatter(np.arange(len(data_receiver)),data_receiver, color = data_receiver_color, lw = 2)
lines = colored_line(np.arange(len(data_receiver)),data_receiver, data_receiver_color, ax_top, linewidth=3, cmap=custom_seismic_softgrey)
ax_top.plot(data_receiver, color = 'k', lw = 2, alpha = 0)
ax_top.plot([0,400],[0,0],color = 'k', lw = 2)
ax_top.axvline(150,c = 'k', ls = '--', lw = 2)
ax_top.axvline(188,c = 'k', ls = '--', lw = 2)
ax_top.axvline(258,c = 'k', ls = '--', lw = 2)
#ax_top.text(155,3.5e-5,'180 ms',color = 'b', size = 14)
#ax_top.text(180,2.5e-5,'440 ms',color = 'r', size = 14)
#ax_top.axvline(np.argmin(np.abs(timecourse_array.data))+1, color = 'k', ls = '--', lw = 2)
#ax_top.axvline(np.argmin(np.abs(timecourse_array.data - np.percentile(wrd_distrib,95)))+1,color = 'k', lw = 2, ls = '--')
#ax_top.axvline(np.argmin(np.abs(timecourse_array.data - 0.3))+1,color = 'k', lw = 2, ls = '--')
ax_top.set_xlim(100,270)
ax_top.spines[['left']].set_linewidth(2)
ax_top.spines[['top', 'right', 'bottom']].set_visible(0)
ax_top.set_ylabel('Received\ninformation\n(bit)', size = 20)
ax_top.tick_params(width=3, labelsize = 16)
ax_top.yaxis.get_offset_text().set_fontsize(16)  
ax_top.set_xticks([])

# Right plot: mean along y
data_right = data_net_filt
data_right = data_net[:,:,180:200]
ax_right = fig.add_subplot(gs[2,1])
ax_right.plot(np.abs(data_right).mean((0,2)), np.linspace(74,0,75), color = 'k', lw = 2)
#ax_right.invert_xaxis()
ax_right.axis('off')


# Colorbar
cax = fig.add_subplot(gs[2,2])
cbar = fig.colorbar(im, cax=cax, format='%.0e')
cbar.ax.set_ylabel('Exchanged information (bit)', fontsize = 14)
cbar.ax.set_yticks([-0.0001,0,0.0001])
cbar.ax.tick_params(labelsize=12)


#Bottom plot
# Top plot: mean along x
ax_bottom = fig.add_subplot(gs[4,0])
data_bottom = data_net_filt
data_bottom = data_net[:,:,:]

data_sender = np.zeros(data_bottom.shape[2])
for i_t in range(data_bottom.shape[2]):
    for i_d in range(data_bottom.shape[1]):
        delay = 75 - i_d
        data_sender[i_t - delay] += data_bottom[:,i_d,i_t].mean() 
data_sender_color = (data_sender - np.min(data_sender)) / (np.max(data_sender) - np.min(data_sender))
ax_bottom.plot([0,400],[0,0],color = 'k', lw = 2)
#ax_bottom.plot(data_receiver, color = 'r', lw = 2)
#ax_bottom.axvline(np.argmin(np.abs(timecourse_array.data))+1, color = 'k', ls = '--', lw = 2)
#ax_bottom.axvline(np.argmin(np.abs(timecourse_array.data - np.percentile(wrd_distrib,95)))+1,color = 'k', lw = 2, ls = '--')
#ax_bottom.axvline(142, color = 'k', ls = '--', lw = 2)
ax_bottom.axvline(115, color = 'k', ls = '--', lw = 2)
ax_bottom.axvline(150, color = 'k', ls = '--', lw = 2)
ax_bottom.axvline(193,color = 'k', lw = 2, ls = '--')
lines = colored_line(np.arange(len(data_sender)),data_sender, data_sender_color, ax_bottom, linewidth=3, cmap=custom_seismic_softgrey)
ax_bottom.plot(data_sender, color = 'grey', lw = 2, alpha = 0)
ax_bottom.spines[['left']].set_linewidth(2)
ax_bottom.spines[['top', 'right', 'bottom']].set_visible(0)
ax_bottom.set_ylabel('Sent\nInformation\n(bit)', size = 20)
ax_bottom.tick_params(width=3, labelsize = 16)
ax_bottom.yaxis.get_offset_text().set_fontsize(16)  
ax_bottom.set_xticks([])

ax_bottom.set_xlim(100,270)

# Ax Filling
ax_fill1 = fig.add_subplot(gs[1,0], sharex = ax_top)
ax_fill2 = fig.add_subplot(gs[3,0], sharex = ax_bottom)
ax_fill1.axis('off')
ax_fill2.axis('off')
#ax_fill1.axvline(np.argmin(np.abs(timecourse_array.data))+1, color = 'k', ls = '--', lw = 2)
#ax_fill1.axvline(np.argmin(np.abs(timecourse_array.data - np.percentile(wrd_distrib,95)))+1,color = 'k', lw = 2, ls = '--')
#ax_fill2.axvline(np.argmin(np.abs(timecourse_array.data))+1, color = 'k', ls = '--', lw = 2)
#ax_fill2.axvline(np.argmin(np.abs(timecourse_array.data - np.percentile(wrd_distrib,95)))+1,color = 'k', lw = 2, ls = '--')
ax_fill1.axvline(188,c = 'k', ls = '--', lw = 2)
ax_fill1.axvline(150,c = 'k', ls = '--', lw = 2)
ax_fill1.axvline(258,c = 'k', ls = '--', lw = 2)
ax_fill2.axvline(115, color = 'k', ls = '--', lw = 2)
ax_fill2.axvline(150, color = 'k', ls = '--', lw = 2)
ax_fill2.axvline(193,color = 'k', lw = 2, ls = '--')


plt.show()
